In [1]:
!pip install groq python-dotenv --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.6 MB/s eta 0:00:00


In [2]:
from dotenv import load_dotenv
import os
import getpass

load_dotenv()

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter GROQ API KEY: ")

Enter GROQ API KEY: ··········


In [3]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

BASE_MODEL = "llama-3.1-8b-instant"

In [4]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": """
You are a Technical Support Expert.
Provide precise, structured, code-focused debugging help.
Explain errors clearly and suggest fixes.
"""
    },

    "billing": {
        "system_prompt": """
You are a Billing Support Expert.
Respond empathetically.
Explain payment issues, refunds, and subscription policies clearly.
"""
    },

    "general": {
        "system_prompt": """
You are a General Customer Support Assistant.
Handle casual conversation and general inquiries politely.
"""
    },

    "tool": {
        "system_prompt": """
You are a Tool Routing Expert.
If financial or real-time data is requested,
use external tools instead of guessing.
"""
    }
}

In [5]:
def route_prompt(user_input):

    routing_prompt = f"""
Classify the following user query into one category:

technical
billing
general
tool

Return ONLY the category name.

Query: {user_input}
"""

    response = client.chat.completions.create(
        model=BASE_MODEL,
        temperature=0,
        messages=[
            {"role": "user", "content": routing_prompt}
        ]
    )

    category = response.choices[0].message.content.strip().lower()
    return category

In [6]:
def get_bitcoin_price():
    return "Current Bitcoin price is approximately $65,000 (mock data)."

In [7]:
def call_expert(category, user_input):

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    response = client.chat.completions.create(
        model=BASE_MODEL,
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content

In [8]:
def process_request(user_input):

    category = route_prompt(user_input)
    print("Routed To:", category)

    if category == "tool":
        return get_bitcoin_price()

    if category not in MODEL_CONFIG:
        category = "general"

    return call_expert(category, user_input)

In [10]:
queries = [
    "My python code gives IndexError on Line 5",
    "I was charged twice a month for my subscription",
    "Hello how are you",
    "What is the name of your pet dog?"
]

for q in queries:
    print("\nUser:", q)
    print("Response:", process_request(q))


User: My python code gives IndexError on Line 5
Routed To: technical
Response: **Error Analysis**

An `IndexError` typically occurs when you're trying to access an element in a list, tuple, or other sequence using an index that is out of range. This usually means that the index you're trying to use is greater than or equal to the length of the sequence.

**Example Code**

To help you better, please provide the code snippet that's causing the error. However, I'll create a simple example to demonstrate how an `IndexError` might occur in Python:

```python
# Example code
my_list = [1, 2, 3]
print(my_list[3])  # This will raise an IndexError
```

**Error Message**

The error message will likely look something like this:

```python
IndexError: list index out of range
```

**Fixing the Error**

To fix the error, you need to ensure that the index you're using is within the valid range of the sequence. Here are a few possible solutions:

1. **Check the length of the sequence**: Before accessi

In [14]:
# =========================
# Bonus Challenge: Tool Use Expert
# =========================

# Mock tool function
def fetch_bitcoin_price():
    # Simulated current BTC price
    return "The current price of Bitcoin is approximately $68,450 (mock data)."

# Tool-use expert
def tool_use_expert(user_query):
    query = user_query.lower()

    if "bitcoin" in query and ("price" in query or "current" in query):
        return fetch_bitcoin_price()

    return "No external tool needed. This query can be handled normally."

# Simple router with tool expert included
def route_query(user_query):
    query = user_query.lower()

    if "bitcoin" in query or "price" in query or "current" in query:
        return "Tool Use Expert"
    elif "code" in query or "python" in query or "program" in query:
        return "Coding Expert"
    elif "math" in query or "solve" in query or "equation" in query:
        return "Math Expert"
    else:
        return "General LLM Expert"

# Final execution
user_input = "What is the current price of Bitcoin?"

selected_expert = route_query(user_input)
print("Selected Expert:", selected_expert)

if selected_expert == "Tool Use Expert":
    answer = tool_use_expert(user_input)
else:
    answer = "Handled by normal LLM response."

print("Response:", answer)

Selected Expert: Tool Use Expert
Response: The current price of Bitcoin is approximately $68,450 (mock data).
